[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-data.ipynb)

# Data Preprocessing & Feature Engineering

*AIBits Academy · Machine Learning End To End · Machine Learning*

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

Some cells on this lesson assume `pandas` and `numpy` are already imported, so we import them once here.

In [ ]:
import numpy as np
import pandas as pd

Raw data is never model-ready. Preprocessing transforms messy real-world data into clean, informative, numerically encoded representations that algorithms can consume.

> **📊 Prerequisite refresher**
>
> Outlier treatment and feature scaling both lean on mean, variance, and standard deviation — if any of those feel rusty, the **Descriptive Statistics** prerequisite page has the full derivations and a worked skewness example before you continue.

## Why Preprocessing Matters

Feature engineering and preprocessing consistently account for the majority of model performance gains in practice. A well-preprocessed dataset with a simple model often outperforms a poorly-preprocessed dataset with a complex model. For Indian fintech, agri-tech, and e-commerce applications, preprocessing quality directly determines model reliability in production.

> **1. Handle Missing Values**
>
> Imputation (mean/median/mode/KNN/MICE) or deliberate removal based on missing mechanism.

> **2. Outlier Treatment**
>
> IQR filtering, Winsorisation, robust scaling — context-dependent.

> **3. Encode Categoricals**
>
> Label, one-hot, ordinal, target, binary encoding — algorithm-dependent.

> **4. Feature Scaling**
>
> StandardScaler, MinMaxScaler, RobustScaler — needed for distance/gradient-based algorithms.

> **5. Feature Selection**
>
> Remove irrelevant/redundant features: correlation filter, RFECV, mutual information, L1 regularisation.

> **6. Feature Creation**
>
> Polynomial features, interaction terms, domain-derived features (days_to_festival, price_per_sqft).

## 1. Handling Missing Values

The strategy depends on the **missing data mechanism**:

| Mechanism | Definition | Example | Strategy |
|---|---|---|---|
| MCAR (Missing Completely At Random) | Missingness unrelated to any variable | Random server glitch drops some rows | Safe to delete rows or impute with mean/median |
| MAR (Missing At Random) | Missingness depends on observed variables | High-income users more likely to skip salary field | Impute using other features (KNN, MICE) |
| MNAR (Missing Not At Random) | Missingness depends on the missing value itself | Loan defaulters omit CIBIL score more often | Model the missingness; add binary "was_missing" indicator feature |

Python — Missing Value Handling (Gujarat Crop Dataset)

```
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer

data = {
    'district': ['Surat', 'Vadodara', 'Rajkot', 'Ahmedabad', 'Anand'],
    'rainfall_mm': [900, np.nan, 750, 680, np.nan],
    'fertiliser_kg': [120, 95, np.nan, 110, 88],
    'yield_qtl': [42, 38, 35, 40, 36]
}
df = pd.DataFrame(data)

# Strategy 1: Mean imputation
mean_imp = SimpleImputer(strategy='mean')
df['rainfall_mm'] = mean_imp.fit_transform(df[['rainfall_mm']]).ravel()

# Strategy 2: KNN imputation (uses feature relationships)
knn_imp = KNNImputer(n_neighbors=2)
num_cols = ['rainfall_mm', 'fertiliser_kg', 'yield_qtl']
df[num_cols] = knn_imp.fit_transform(df[num_cols])

# Strategy 3: MNAR indicator — add "was_missing" binary flag
df['fertiliser_was_missing'] = df['fertiliser_kg'].isnull().astype(int)
print(df)
```

## 2. Outlier Detection & Treatment

$$\text{IQR} = Q_3-Q_1 \quad\big|\quad \text{Lower fence} = Q_1-1.5\times\text{IQR} \quad\big|\quad \text{Upper fence} = Q_3+1.5\times\text{IQR}$$

Python — IQR + Z-score + Winsorisation

```
from scipy import stats

prices = pd.Series([45, 52, 48, 51, 49, 650, 47, 53, 3, 50])  # ₹ lakhs

Q1, Q3 = prices.quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers_iqr = prices[(prices < Q1-1.5*IQR) | (prices > Q3+1.5*IQR)]
print("IQR outliers:", outliers_iqr.tolist())

z_scores = np.abs(stats.zscore(prices))
print("Z-score outliers:", prices[z_scores > 2.5].tolist())

prices_winsorised = prices.clip(prices.quantile(0.05), prices.quantile(0.95))
print("Winsorised:", prices_winsorised.tolist())
```

### Visualising the IQR Fences

Every point on this number line is one of the ten ₹-lakh prices above — the shaded band is the "normal" zone; anything outside the dashed fences gets flagged.

## 3. Encoding Categorical Variables

| Encoding | When to Use | Caution |
|---|---|---|
| **One-Hot** | Nominal categories (city, fabric type) | Creates many columns for high-cardinality features; dummy variable trap |
| **Label** | Tree-based models only; ordinal data | Implies false ordinal relationship in linear models |
| **Ordinal** | Ordered categories (Low, Medium, High) | Must specify correct order explicitly |
| **Target** | High-cardinality nominal (pincode, city with 500 cities) | Causes leakage if not done inside CV folds |

Python — One-Hot & Ordinal Encoding (Mumbai Loan Dataset)

```
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df_loan = pd.DataFrame({
    'employment_type': ['Salaried','Self-Employed','Salaried','Business','Self-Employed'],
    'credit_grade': ['A','C','B','A','D'],   # ordinal: A > B > C > D
    'income_lakhs': [8.5, 12.0, 6.2, 25.0, 9.8]
})

# One-hot encode employment_type (nominal)
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids dummy trap
emp_enc = ohe.fit_transform(df_loan[['employment_type']])
print(pd.DataFrame(emp_enc, columns=ohe.get_feature_names_out(['employment_type'])))

# Ordinal encode credit_grade
oe = OrdinalEncoder(categories=[['A','B','C','D']])
df_loan['grade_num'] = oe.fit_transform(df_loan[['credit_grade']])
print(df_loan[['credit_grade', 'grade_num']])
```

## 4. Feature Scaling

| Scaler | Formula | Best For | Caution |
|---|---|---|---|
| `StandardScaler` | x' = (x − μ) / σ | Most algorithms; approximately normal features | Not robust to outliers |
| `MinMaxScaler` | x' = (x − min) / (max − min) | Neural networks; bounded output [0,1] | Severely distorted by outliers |
| `RobustScaler` | x' = (x − Q2) / IQR | Data with many outliers (medical, financial) | Output range not bounded |
| Log transform | x' = log(x + 1) | Right-skewed data (income, property prices) | Requires x ≥ 0; undo on predictions |

> **⚠️ When NOT to Scale**
>
> Tree-based algorithms (Decision Trees, Random Forest, XGBoost) are invariant to monotonic feature transformations — they only care about split thresholds, not absolute values. Scaling is unnecessary for these models but harmless to include in a pipeline for consistency.

## 5. Complete sklearn Pipeline — Ahmedabad Real Estate

Python — Full Preprocessing Pipeline

```
import pandas as pd; import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

np.random.seed(42); n = 200
df = pd.DataFrame({
    'area_sqft': np.random.randint(600, 3000, n),
    'bedrooms': np.random.choice([1,2,3,4], n),
    'age_years': np.random.randint(0, 30, n).astype(float),
    'locality': np.random.choice(['Satellite','Prahlad Nagar','Navrangpura','Bopal'], n),
    'furnished': np.random.choice(['Yes','No','Semi'], n),
    'floor': np.random.randint(0, 20, n).astype(float)
})
df.loc[df.sample(20).index, 'age_years'] = np.nan
df['price_lakhs'] = (0.04*df['area_sqft'] + 3*df['bedrooms']
                      - 0.5*df['age_years'].fillna(10) + np.random.normal(0,5,n))

X = df.drop('price_lakhs', axis=1); y = df['price_lakhs']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_features = ['area_sqft', 'bedrooms', 'age_years', 'floor']
cat_features = ['locality', 'furnished']

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_pipe, num_features),
                                   ('cat', cat_pipe, cat_features)])

full_pipe = Pipeline([('prep', preprocessor), ('model', LinearRegression())])
full_pipe.fit(X_train, y_train)
print(f"R² on test set: {full_pipe.score(X_test, y_test):.4f}")
```

## Cleaning Messy Text Columns — pandas String Methods

Real Indian business data is rarely clean: inconsistent casing, stray whitespace, mixed formats ("₹1,20,000" vs "120000"). pandas' vectorised `.str` accessor handles this far faster than row-by-row Python loops:

In [ ]:
# Messy Zomato restaurant names & prices scraped from multiple sources
raw = pd.DataFrame({
    'name': ['  Punjabi Tadka ', 'CAFE COFFEE DAY', 'sagar-Ratna', 'Barbeque Nation  '],
    'price_text': ['₹1,200 for two', 'Rs. 450', '₹850', '2,500 INR']
})
# Strip whitespace, standardise casing
raw['name_clean'] = raw['name'].str.strip().str.title().str.replace('-', ' ', regex=False)
# Extract numeric price with regex, coerce to float
raw['price_inr'] = (raw['price_text']
                    .str.replace(r'[^\d]', '', regex=True)
                    .astype(int))
print(raw[['name_clean','price_inr']])

> **💡 Going Deeper**
>
> Cleaning and encoding are only half the story — **constructing** new, more informative features (aggregations across related rows, datetime-derived signals, interaction terms) usually moves model accuracy far more than any encoding choice. The next chapter, **Feature Engineering**, covers this in depth using pandas' `groupby`, merge, and reshape tools.

## 6. Feature Selection

| Method | Approach | When to Use |
|---|---|---|
| Correlation filter | Drop features with \|corr with target\| < threshold; drop one from each pair with \|corr\| > 0.9 | Quick baseline; numerical features |
| Mutual Information | Measures statistical dependency; handles non-linear relationships | Mixed feature types; non-linear targets |
| RFECV | Recursive Feature Elimination with CV — fits model, removes least important feature, repeats | When compute budget allows; works with any model |
| L1 (Lasso) | Drives unimportant feature weights to exactly zero | Linear models; high-dimensional sparse features |
| Tree importance | Random Forest / XGBoost built-in importance scores | After fitting a tree-based model |

> **📋 Real-World Case Study — House Price Prediction (Ames, Iowa)**
>
> A real 2,930-row housing dataset had **Pool QC missing in 99.56% of rows** and **Misc Feature missing in 96.38%** — well past the point where deleting the column looks like the obvious call. Cross-checking against `Pool Area` (genuinely 0 for all 2,917 of those rows) and `Misc Val` (genuinely 0 for the matching rows) confirmed the missingness wasn't random data loss at all — it meant "this house doesn't have one." Recoding both as an explicit "No Pool" / "No feature" category preserved real signal that outright deletion would have silently thrown away.

> **🔗 Real-World Link — Handling Missing Values Using Linear Regression**
>
> A 414-row Taiwan real-estate dataset uses a regression model itself to predict and fill in missing house prices — regression as an imputation tool, not just a prediction target. [See the case study →](https://statso.io/handling-missing-values-using-linear-regression-case-study-990bf1f2c74f) ·

## Extracting Structure from Text — `str.extract`

Beyond cleaning text, you often need to *pull structured fields out of* a messy string column using a regex with capture groups. `Series.str.extract` turns each capture group into its own new column — a fast way to manufacture several features from one raw text field:

In [ ]:
import pandas as pd
s = pd.Series(['Order #A1023 - Surat', 'Order #B2045 - Mumbai', 'Order #C3087 - Pune'])
# three capture groups → three new columns
parts = s.str.extract(r'#([A-Z])(\d+)\s*-\s*(\w+)')
parts.columns = ['batch', 'order_no', 'city']
print(parts)

## 7. L2 Normalisation — Scaling Rows to Unit Length

The scaling methods above (Standardisation, Min-Max) work *column by column* — each feature is rescaled independently. **L2 normalisation** is different: it rescales each *row* so that the row's vector length (its Euclidean/L2 norm) becomes exactly 1. This is used when the *direction* of a feature vector matters more than its magnitude — for example comparing customer behaviour profiles regardless of overall activity volume, or before cosine-similarity-based methods.

In [ ]:
from sklearn.preprocessing import normalize
import numpy as np

# Each row = one Flipkart order: [items, total_qty, distance_km]
X = np.array([[2, 5, 8.0], [1, 3, 4.0], [6, 12, 20.0]])
X_l2 = normalize(X, norm='l2')
print(np.round(X_l2, 4))
print("row lengths:", np.round(np.linalg.norm(X_l2, axis=1), 4))

Every row now has length 1, so only the *relative proportions* of items-to-quantity-to-distance survive — a big order and a small order with the same shape become identical vectors. Contrast that with column scaling, which preserves the difference between big and small orders.

## 8. One-Hot vs Label Encoding — Which and When

Section 3 above introduced encoding categorical variables. It's worth being explicit about the two dominant choices and their trap. **Label encoding** maps each category to an integer (Bengaluru→0, Mumbai→1, Surat→2). **One-hot encoding** creates a separate 0/1 column per category.

In [ ]:
import pandas as pd
cities = pd.Series(['Surat', 'Mumbai', 'Surat', 'Bengaluru', 'Mumbai'])

# Label encoding — compact, but invents a fake ordering 0 < 1 < 2
labels = cities.astype('category').cat.codes
# One-hot encoding — no fake ordering, one column per city
onehot = pd.get_dummies(cities, prefix='city').astype(int)
print(onehot)

> **⚠ The Label-Encoding Trap**
>
> Label encoding on a *nominal* variable (cities, payment methods, colours) silently tells the model that Surat (2) is "greater than" Mumbai (1) — an ordering that doesn't exist. Linear models and distance-based models (KNN, K-Means) will act on that phantom ordering and mislead. Use label encoding only for genuinely **ordinal** variables (Low<Medium<High) or for tree models that don't assume order; use **one-hot** for nominal variables. The cost of one-hot is width: a 100-category column becomes 100 columns (high cardinality → consider target/frequency encoding instead).

## 9. Feature Selection — Filter, Wrapper & Embedded

Section 6 above showed automatic feature selection in passing. The three *families* of feature-selection methods are worth distinguishing, because they trade off speed against accuracy very differently:

| Family | How it works | Examples | Cost |
|---|---|---|---|
| **Filter** | Score each feature with a statistic, independent of any model; keep the top-scoring ones | SelectKBest (ANOVA F, chi², mutual information), correlation | Cheapest — no model trained |
| **Wrapper** | Search subsets of features, training a model on each to score it | RFE, forward selection, backward elimination | Expensive — many model fits |
| **Embedded** | The model selects features *as part of* its own fitting | LASSO (LassoCV), Random-Forest / XGBoost feature importance | Moderate — one model fit |

Here all three families are pointed at the same synthetic dataset where only 2 of 5 features (`f3`, `f4`) actually carry signal — a good test of whether each method can find the needles:

In [ ]:
from sklearn.datasets import make_regression
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor

# 5 features, but only 2 are truly informative
X, y = make_regression(n_samples=200, n_features=5, n_informative=2,
                       noise=10, random_state=42)

# FILTER: SelectKBest with ANOVA F-score
kbest = SelectKBest(score_func=f_regression, k=2).fit(X, y)
print("Filter (SelectKBest) top-2:", np.where(kbest.get_support())[0])

# EMBEDDED: LASSO drives irrelevant coefficients to exactly zero
lasso = LassoCV(cv=5, random_state=42).fit(X, y)
print("LASSO nonzero coefs:", np.round(lasso.coef_, 2))

# EMBEDDED: Random-Forest importances
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y)
print("RF importances:", np.round(rf.feature_importances_, 3))

All three families independently zero in on `f3` and `f4` — the only features with real signal. LASSO shrinks the three noise features' coefficients to essentially zero; Random Forest assigns them tiny importances (≈0.04 each vs 0.33 and 0.55); the filter's F-test ranks them out. When methods from three different families agree, that's strong evidence you've found the genuinely useful features.

> **⚠ Common Preprocessing Pitfalls**
>
> A few mistakes that quietly corrupt results: **(1) Data leakage** — fitting a scaler or imputer on the *full* dataset before the train/test split lets test-set statistics leak into training; always `fit` on train only, then `transform` both (this is exactly why the sklearn `Pipeline` in Section 5 matters). **(2) Scaling categorical/one-hot columns** — standardising a 0/1 indicator column is meaningless and distorts it; scale only genuine numeric features. **(3) Ignoring the distribution** — Standardisation assumes a roughly bell-shaped feature; on heavily skewed data a log or power transform first is often better. **(4) Forgetting outliers** — Min-Max scaling is extremely sensitive to a single extreme value; consider RobustScaler when outliers are present.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Turn messy price text into numbers

Scraped prices arrive as text: `"₹1,200"`, `"Rs. 450"`, `"2,500 INR"`. Use a regular expression (`Series.str.extract`) to store the numbers as integers in `prices`.

In [ ]:
import pandas as pd
raw = pd.Series(["₹1,200", "Rs. 450", "2,500 INR"])
prices = None   # TODO


In [ ]:
try:
    check("integers extracted", prices is not None and list(prices) == [1200, 450, 2500])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
raw = pd.Series(["₹1,200", "Rs. 450", "2,500 INR"])
prices = raw.str.extract(r"(\d[\d,]*)")[0].str.replace(",", "").astype(int)

```

</details>

### Exercise 2 · Medium · One-hot encode without inventing an order

Encode `city` as indicator columns with `pd.get_dummies` (integers 0/1, prefix `city`) into `onehot`. Why is this safer than label codes for a linear model?

In [ ]:
import pandas as pd
city = pd.Series(["Surat", "Mumbai", "Surat", "Pune", "Mumbai"])
onehot = None   # TODO


In [ ]:
try:
    check("3 columns", onehot is not None and onehot.shape == (5, 3))
    check("exactly one 1 per row", (onehot.sum(axis=1) == 1).all())
    check("column names", sorted(onehot.columns) == ["city_Mumbai", "city_Pune", "city_Surat"])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
city = pd.Series(["Surat", "Mumbai", "Surat", "Pune", "Mumbai"])
onehot = pd.get_dummies(city, prefix="city").astype(int)

```

Label codes (0, 1, 2) tell a linear model that Pune is 'twice' Mumbai; indicator columns make no such claim.

</details>

### Exercise 3 · Stretch · L2-normalise rows by hand

Write `l2_normalize(X)` that scales every **row** of a 2-D array to unit Euclidean length, without `sklearn`. It must match `sklearn.preprocessing.normalize`.

In [ ]:
import numpy as np
def l2_normalize(X):
    pass   # TODO


In [ ]:
try:
    from sklearn.preprocessing import normalize
    X = np.array([[2, 5, 8.0], [1, 3, 4.0], [6, 12, 20.0]])
    check("matches sklearn", np.allclose(l2_normalize(X), normalize(X, norm="l2")))
    check("rows have length 1", np.allclose(np.linalg.norm(l2_normalize(X), axis=1), 1))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def l2_normalize(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

```

</details>

---
*Back to the course: **Machine Learning End To End → Data Preprocessing & Feature Engineering**.*